# 🧭 Chapter 07-02a: Control Flow — Cycles, Checkpoints & Human-in-the-Loop

---

## 📖 The Story: The Editor Who Keeps Sending Drafts Back

You write a report. Your editor reads it, marks corrections, and sends it back. You revise. They review again. This loop continues until they say *"Approved!"*

That's a **cycle** in a graph. And sometimes the editor wants to **pause** the process and check with the CEO before publishing — that's **human-in-the-loop**.

---

## 🎯 Cycles in LangGraph

```
╔══════════════════════════════════════════════════════════════════════╗
║  CYCLES (LOOPS)                                                     ║
║                                                                      ║
║  A cycle is when an edge points BACK to an earlier node.           ║
║  This is what makes agents powerful — they can retry, refine,      ║
║  and iterate until they get it right.                               ║
║                                                                      ║
║  Agent → Tool → Agent → Tool → Agent → END                        ║
║  (The Agent↔Tool cycle repeats until done)                         ║
╚══════════════════════════════════════════════════════════════════════╝
```

```
    ┌──────────┐
    │  Agent   │◄──────────┐
    └────┬─────┘           │
         │                 │
    ┌────▼─────┐     ┌────┴─────┐
    │ Should   │ Yes │  Tools   │
    │ use tool?├─────►          │
    └────┬─────┘     └──────────┘
         │ No
    ┌────▼─────┐
    │   END    │
    └──────────┘
```

---

## 🔖 Checkpointing — Save & Resume

```
╔══════════════════════════════════════════════════════════════════════╗
║  CHECKPOINTING                                                      ║
║                                                                      ║
║  Save the graph's state at any point so you can:                   ║
║  1. Resume later (long-running workflows)                          ║
║  2. Branch from a checkpoint ("what-if" scenarios)                 ║
║  3. Debug step-by-step what went wrong                             ║
║                                                                      ║
║  Backends: MemorySaver (dev), SqliteSaver, PostgresSaver (prod)    ║
╚══════════════════════════════════════════════════════════════════════╝
```

```python
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# Each conversation thread gets its own state
config = {"configurable": {"thread_id": "user-123"}}
app.invoke({"messages": [HumanMessage("Hi")]}, config)
```

---

## 🧑 Human-in-the-Loop (HITL)

Three patterns:

### 1. Interrupt Before — Pause for approval
```python
app = graph.compile(
    checkpointer=memory,
    interrupt_before=["execute_action"]
)
# Pauses BEFORE execute_action runs.
# Human reviews, then resumes with app.invoke(None, config)
```

### 2. Interrupt After — Review results
```python
app = graph.compile(
    checkpointer=memory,
    interrupt_after=["generate_plan"]
)
# Pauses AFTER generate_plan. Human can edit the plan before continuing.
```

### 3. State Editing — Human modifies state
```python
# After interrupt, update state before resuming
app.update_state(config, {"plan": "modified plan"})
app.invoke(None, config)  # Continue with modified state
```

---

## 🧩 Subgraphs — Graphs Within Graphs

```python
# A subgraph is just a compiled graph used as a node
research_graph = StateGraph(ResearchState)
# ... add nodes, edges ...
research_app = research_graph.compile()

# Use it as a node in the parent graph
parent_graph.add_node("research", research_app)
```

**Use cases**: Modular agent design, team of specialists, complex workflows broken into sub-workflows.

---

## 🎓 Interview Corner

**Q: How would you implement a human-in-the-loop approval step in a LangGraph agent?**

<details>
<summary>💡 Click for Expert Answer</summary>

1. **Compile with interrupt**: `graph.compile(checkpointer=memory, interrupt_before=["execute"])`
2. **First invoke**: Agent plans the action, hits the interrupt, pauses
3. **Human review**: Read the planned action from state via `app.get_state(config)`
4. **Approve**: Resume with `app.invoke(None, config)`
5. **Reject/Modify**: Update state with `app.update_state(config, new_state)` then resume
6. **Production**: Use PostgresSaver for persistence, webhook for human notification

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Cycles = edges back to earlier nodes (agent↔tool loop).        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Checkpointing saves state — MemorySaver (dev), Postgres (prod).│
├─────────────────────────────────────────────────────────────────────┤
│  📌 interrupt_before/after pauses the graph for human review.      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 update_state() lets humans modify state before resuming.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Subgraphs = compiled graphs used as nodes in parent graphs.    │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On — Agent That Searches, Decides & Acts with Cycles!

---

### 🎉 Cycles & HITL Mastered!
The real power of graphs — loops, checkpoints, and human oversight. 🚀